# Distribuciones sin densidad

En el curso de probabilidad clásico, casi siempre trabajamos con distribuciones que tienen **función de densidad** (continuas) o **función de probabilidad puntual** (discretas). Pero el teorema de descomposición de Lebesgue nos dice que existe una tercera clase: las distribuciones **singulares continuas**.

Una distribución singular continua tiene las siguientes propiedades:

- Su función de distribución acumulada $F(x)$ es **continua** (no tiene saltos).
- Sin embargo, $F'(x) = 0$ en casi todo punto (en el sentido de la medida de Lebesgue).
- No tiene densidad: no existe ninguna función $f$ tal que $F(x) = \int_{-\infty}^x f(t)\, dt$.
- Toda la "masa" está concentrada en un conjunto de medida de Lebesgue **cero**.

> **En criollo:** la distribución es continua (no asigna probabilidad a ningún punto individual), pero tampoco se puede describir con una densidad porque su masa está apoyada en un conjunto "infinitamente poroso", como el conjunto de Cantor. La función crece, pero lo hace en un conjunto tan raro que su derivada es cero casi en todas partes.

---
## 1. Distribución de Cantor

### Definición formal

El **conjunto de Cantor** $\mathcal{C}$ se construye eliminando iterativamente el tercio central de cada intervalo, partiendo de $[0,1]$:

$$\mathcal{C} = \left\{ x \in [0,1] : x = \sum_{k=1}^{\infty} \frac{a_k}{3^k},\ a_k \in \{0, 2\} \right\}$$

Es decir, son los números de $[0,1]$ cuya expansión en base 3 **no contiene el dígito 1**.

La **distribución de Cantor** es la medida de probabilidad $\mu_C$ apoyada en $\mathcal{C}$, definida como el límite de distribuciones uniformes sobre los intervalos que sobreviven en cada etapa de la construcción. Su FDA (función de distribución acumulada) es la famosa **función escalera del diablo** (*devil's staircase*):

$$F_C(x) = \lim_{n \to \infty} F_n(x)$$

donde $F_n$ es la FDA de la distribución uniforme sobre los $2^n$ intervalos de la etapa $n$.

### Propiedades
- $\mathcal{C}$ tiene medida de Lebesgue cero pero cardinalidad $|\mathcal{C}| = |\mathbb{R}|$.
- $F_C$ es continua, no decreciente, $F_C(0)=0$, $F_C(1)=1$.
- $F_C'(x) = 0$ para casi todo $x$ (en los intervalos removidos, que tienen medida 1).
- No existe densidad $f$ tal que $F_C(x) = \int_0^x f(t)\,dt$.

### En criollo

> Imaginá que tenés un kilo de harina y lo distribuís sobre la mesa. Ahora quitás el tercio del medio, luego el tercio del medio de lo que queda a izquierda y a derecha, y así infinitamente. Lo que queda (el conjunto de Cantor) parece "polvo": tiene infinitos puntos, pero ocupa longitud cero. La distribución de Cantor pone toda su masa en ese polvo. La FDA crece, pero lo hace **en escalones infinitesimales** que nunca podés ver individualmente: subís, subís, pero nunca en ningún punto concreto.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Construcción de la función escalera del diablo ──────────────────────────
def cantor_fda(x, n_iter=12):
    """Aproxima la FDA de Cantor evaluando x ∈ [0,1] con n_iter iteraciones."""
    x = np.asarray(x, dtype=float)
    result = np.zeros_like(x)
    lo = np.zeros_like(x)
    hi = np.ones_like(x)
    val_lo = np.zeros_like(x)

    for _ in range(n_iter):
        mid = (lo + hi) / 2
        left  = (lo + mid) / 2   # tercio izquierdo
        right = (mid + hi) / 2   # tercio derecho
        mid_val = (val_lo + (val_lo + (hi - lo))) / 2  # valor en el tercio central

        # En el tercio central la función es constante = mid_val
        in_middle = (x >= left) & (x <= right)
        result = np.where(in_middle, (val_lo + 1 - val_lo) / 2 + val_lo, result)

        # Actualizar para rama izquierda y derecha
        go_right = x > right
        lo  = np.where(go_right, right, lo)
        val_lo = np.where(go_right, val_lo + (1 - val_lo - val_lo) / 2 + val_lo, val_lo)
        hi  = np.where(~go_right & ~in_middle, left, hi)

    return result

# Alternativa más limpia: iteración directa en base 3
def devil_staircase(x_vals, depth=15):
    """Escalera del diablo via algoritmo iterativo."""
    out = []
    for x in x_vals:
        val = 0.0
        lo, hi = 0.0, 1.0
        for _ in range(depth):
            m1 = lo + (hi - lo) / 3
            m2 = lo + 2 * (hi - lo) / 3
            if x < m1:
                hi = m1
            elif x > m2:
                val += 1 / 2**(_ + 1)  # escalón
                lo = m2
            else:
                val += 1 / 2**(_ + 1)
                break
        out.append(val)
    return np.array(out)

x = np.linspace(0, 1, 3000)
fda_cantor = devil_staircase(x, depth=14)

# ── Construcción iterativa del conjunto de Cantor (visualización) ────────────
def cantor_intervals(n):
    """Devuelve los intervalos que sobreviven después de n iteraciones."""
    intervals = [(0.0, 1.0)]
    for _ in range(n):
        new = []
        for a, b in intervals:
            third = (b - a) / 3
            new.append((a, a + third))
            new.append((b - third, b))
        intervals = new
    return intervals

# ── Plot ─────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(12, 8))
fig.patch.set_facecolor('#0f0f0f')
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

ax1 = fig.add_subplot(gs[0, :])
ax2 = fig.add_subplot(gs[1, 0])
ax3 = fig.add_subplot(gs[1, 1])

for ax in [ax1, ax2, ax3]:
    ax.set_facecolor('#1a1a1a')
    ax.tick_params(colors='#aaaaaa', labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#444444')

# FDA: escalera del diablo
ax1.plot(x, fda_cantor, color='#e8b96a', lw=1.8, label='FDA de Cantor')
ax1.set_title('Función de distribución acumulada de Cantor\n(Escalera del Diablo)', 
              color='#e8b96a', fontsize=11, pad=10)
ax1.set_xlabel('x', color='#888888', fontsize=9)
ax1.set_ylabel('F(x)', color='#888888', fontsize=9)
ax1.legend(fontsize=8, facecolor='#2a2a2a', edgecolor='#555555', labelcolor='#cccccc')
ax1.grid(True, alpha=0.15, color='#666666')

# Construcción iterativa
colors = ['#c0392b', '#e67e22', '#f1c40f', '#2ecc71', '#3498db', '#9b59b6']
for i in range(6):
    intervals = cantor_intervals(i)
    for a, b in intervals:
        ax2.barh(i, b - a, left=a, height=0.6, color=colors[i], alpha=0.85)
ax2.set_title('Construcción iterativa\ndel conjunto de Cantor', 
              color='#e8b96a', fontsize=10, pad=8)
ax2.set_xlabel('x', color='#888888', fontsize=9)
ax2.set_ylabel('Iteración', color='#888888', fontsize=9)
ax2.set_yticks(range(6))
ax2.set_yticklabels([f'n={i}' for i in range(6)], color='#aaaaaa', fontsize=8)
ax2.set_xlim(0, 1)
ax2.grid(True, alpha=0.1, color='#666666', axis='x')

# Derivada numérica (casi cero en todas partes)
dx = x[1] - x[0]
deriv = np.gradient(fda_cantor, dx)
ax3.plot(x, deriv, color='#e74c3c', lw=0.8, alpha=0.8)
ax3.set_title('Derivada numérica de F(x)\n(≈ 0 casi en todas partes)', 
              color='#e8b96a', fontsize=10, pad=8)
ax3.set_xlabel('x', color='#888888', fontsize=9)
ax3.set_ylabel("F'(x)", color='#888888', fontsize=9)
ax3.grid(True, alpha=0.15, color='#666666')
ax3.set_ylim(-0.5, max(deriv) * 1.1)

plt.suptitle('Distribución de Cantor', color='#ffffff', fontsize=13, fontweight='bold', y=1.01)
plt.savefig('/tmp/cantor.png', dpi=130, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('Nota: la derivada numérica muestra picos artificiales en los bordes de los escalones.')
print('Matemáticamente, F\'(x) = 0 en casi todo punto.')

---
## 2. Escalera del Diablo como distribución (enfoque directo)

### Definición formal

La **función de Cantor-Lebesgue** $\varphi: [0,1] \to [0,1]$ se define como:

$$\varphi(x) = \begin{cases}
\frac{1}{2} & x \in \left(\frac{1}{3}, \frac{2}{3}\right) \\
\frac{1}{4} & x \in \left(\frac{1}{9}, \frac{2}{9}\right) \\
\frac{3}{4} & x \in \left(\frac{7}{9}, \frac{8}{9}\right) \\
\vdots
\end{cases}$$

y se extiende por continuidad a todo $[0,1]$. Podemos verla como FDA de una variable aleatoria $X$ tal que:

$$X = \sum_{k=1}^{\infty} \frac{B_k}{2^k}$$

donde $B_k \sim \text{Bernoulli}(1/2)$ son i.i.d. y los dígitos en base 3 de $X$ son $a_k = 2B_k \in \{0, 2\}$.

### Propiedad clave

Si $X \sim \mu_C$ (distribución de Cantor), entonces:
$$\mathbb{E}[X] = \frac{1}{2}, \quad \text{Var}(X) = \frac{1}{8}$$

A pesar de no tener densidad, los momentos están perfectamente definidos.

### En criollo

> Tirás una moneda infinitas veces. Si sale cara, anotás un 2; si sale cruz, anotás un 0. Formás el número $0.a_1 a_2 a_3\ldots$ en base 3. Ese número siempre cae en el conjunto de Cantor. La distribución resultante es la de Cantor, y la FDA es la escalera del diablo. En cada lanzamiento "subís" o "te quedás" en la escalera, pero el salto se hace infinitamente chico.

In [ ]:
# Simulación de X ~ Cantor via lanzamientos de moneda en base 3
np.random.seed(42)
N = 50_000
K = 20  # iteraciones

# Cada fila: K lanzamientos de Bernoulli(1/2)
B = np.random.randint(0, 2, size=(N, K))  # 0 o 1
digits = 2 * B  # dígitos en base 3: 0 o 2
potencias = 3.0 ** (-np.arange(1, K + 1))
X_cantor = digits @ potencias  # suma: X = Σ aₖ/3ᵏ

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.patch.set_facecolor('#0f0f0f')

for ax in axes:
    ax.set_facecolor('#1a1a1a')
    ax.tick_params(colors='#aaaaaa', labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#444444')

# Histograma de la muestra
axes[0].hist(X_cantor, bins=200, density=True, color='#3498db', alpha=0.8, edgecolor='none')
axes[0].set_title('Histograma de 50.000 muestras\nX ~ Cantor (base 3, dígitos {0,2})',
                  color='#e8b96a', fontsize=9)
axes[0].set_xlabel('x', color='#888888')
axes[0].set_ylabel('Densidad empírica', color='#888888')
axes[0].text(0.5, 0.92, '← Sin densidad: los "vacíos" son los tercios removidos',
             transform=axes[0].transAxes, color='#e74c3c', fontsize=7, ha='center')

# FDA empírica
X_sorted = np.sort(X_cantor)
y_ecdf = np.arange(1, N + 1) / N
axes[1].plot(X_sorted, y_ecdf, color='#e8b96a', lw=0.8)
axes[1].set_title('FDA empírica\n(replica la escalera del diablo)',
                  color='#e8b96a', fontsize=9)
axes[1].set_xlabel('x', color='#888888')
axes[1].set_ylabel('F̂(x)', color='#888888')
axes[1].grid(True, alpha=0.15, color='#666666')

# Momentos
media_teo = 0.5
var_teo   = 1/8
axes[2].axis('off')
tabla = [
    ['', 'Teórico', 'Empírico'],
    ['E[X]', '0.5000', f'{X_cantor.mean():.4f}'],
    ['Var(X)', '0.1250', f'{X_cantor.var():.4f}'],
    ['Mediana', '0.5000', f'{np.median(X_cantor):.4f}'],
    ['Mín', '0.0000', f'{X_cantor.min():.4f}'],
    ['Máx', '1.0000', f'{X_cantor.max():.4f}'],
]
t = axes[2].table(cellText=tabla[1:], colLabels=tabla[0],
                  loc='center', cellLoc='center')
t.auto_set_font_size(False)
t.set_fontsize(9)
for (r, c), cell in t.get_celld().items():
    cell.set_facecolor('#1a1a1a' if r > 0 else '#2a2a2a')
    cell.set_text_props(color='#cccccc')
    cell.set_edgecolor('#444444')
axes[2].set_title('Momentos teóricos vs. empíricos', color='#e8b96a', fontsize=9, pad=60)

plt.suptitle('Simulación: X ~ Distribución de Cantor', color='#ffffff', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/cantor_sim.png', dpi=130, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

---
## 3. Mezcla discreta-continua (distribución mixta)

### Definición formal

Una **distribución mixta** combina una parte discreta y una parte continua. No es estrictamente singular, pero tampoco tiene densidad en el sentido estándar. Si $p \in (0,1)$, definimos:

$$F(x) = p \cdot F_{\text{disc}}(x) + (1-p) \cdot F_{\text{cont}}(x)$$

Por el **Teorema de Lebesgue-Radon-Nikodym**, cualquier medida de probabilidad $\mu$ sobre $\mathbb{R}$ se descompone de forma única:

$$\mu = \mu_{ac} + \mu_{sc} + \mu_{pp}$$

donde $\mu_{ac}$ es absolutamente continua (tiene densidad), $\mu_{sc}$ es singular continua (como Cantor), y $\mu_{pp}$ es puntual pura (discreta).

Una distribución **mixta** tiene $\mu_{ac} \neq 0$ y $\mu_{pp} \neq 0$, con $\mu_{sc} = 0$.

**Ejemplo canónico:** tiempo hasta un evento que puede ocurrir en $t=0$ con probabilidad $p$ o seguir una exponencial con parámetro $\lambda$:

$$F(x) = p \cdot \mathbf{1}_{x \geq 0} + (1-p)(1 - e^{-\lambda x}) \cdot \mathbf{1}_{x \geq 0}$$

Esta distribución **no tiene densidad** porque $P(X=0) = p > 0$, pero tampoco es puramente discreta.

### En criollo

> Pensá en el tiempo que tarda un cliente en llamar a un call center. Con probabilidad $p$ llama instantáneamente (en $t=0$, quizás ya estaba esperando). Si no, espera un tiempo aleatorio que sigue una exponencial. El resultado no es ni discreto ni continuo puro: tiene un "átomo" en $t=0$ y una cola continua. No podés escribirlo con una sola densidad.

In [ ]:
from scipy import stats

p = 0.35       # probabilidad del átomo en 0
lam = 1.5      # parámetro exponencial
N = 30_000
np.random.seed(0)

# Simulación
atomos  = np.random.binomial(1, p, N)          # 1 = cae en t=0
continuos = np.random.exponential(1/lam, N)    # parte continua
X_mix = np.where(atomos == 1, 0.0, continuos)

x_grid = np.linspace(-0.1, 4, 1000)
fda_mix = p * (x_grid >= 0) + (1 - p) * stats.expon.cdf(x_grid, scale=1/lam) * (x_grid >= 0)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.patch.set_facecolor('#0f0f0f')
for ax in axes:
    ax.set_facecolor('#1a1a1a')
    ax.tick_params(colors='#aaaaaa', labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#444444')

# Histograma (sin el átomo)
X_cont_only = X_mix[X_mix > 0]
axes[0].hist(X_cont_only, bins=80, density=True, color='#2ecc71', alpha=0.75, label='Parte continua')
axes[0].axvline(0, color='#e74c3c', lw=2, label=f'Átomo en 0 (p={p})')
axes[0].set_title('Histograma parte continua\n+ átomo en t=0', color='#e8b96a', fontsize=9)
axes[0].set_xlabel('x', color='#888888')
axes[0].legend(fontsize=7, facecolor='#2a2a2a', edgecolor='#555', labelcolor='#ccc')
axes[0].set_xlim(-0.2, 4)

# FDA teórica
axes[1].plot(x_grid, fda_mix, color='#e8b96a', lw=2)
axes[1].axvline(0, color='#e74c3c', lw=1, ls='--', alpha=0.6)
# Salto en x=0
axes[1].annotate(f'Salto = p = {p}', xy=(0, p), xytext=(0.5, p - 0.1),
                 arrowprops=dict(arrowstyle='->', color='#e74c3c'),
                 color='#e74c3c', fontsize=8)
axes[1].set_title('FDA de la distribución mixta\n(salto en t=0)', color='#e8b96a', fontsize=9)
axes[1].set_xlabel('x', color='#888888')
axes[1].set_ylabel('F(x)', color='#888888')
axes[1].grid(True, alpha=0.15, color='#666666')

# FDA empírica vs teórica
X_sorted = np.sort(X_mix)
y_ecdf = np.arange(1, N+1) / N
axes[2].plot(X_sorted, y_ecdf, color='#3498db', lw=0.8, label='FDA empírica', alpha=0.8)
axes[2].plot(x_grid, fda_mix, color='#e8b96a', lw=1.5, ls='--', label='FDA teórica')
axes[2].set_title('FDA empírica vs. teórica', color='#e8b96a', fontsize=9)
axes[2].set_xlabel('x', color='#888888')
axes[2].set_xlim(-0.2, 4)
axes[2].legend(fontsize=7, facecolor='#2a2a2a', edgecolor='#555', labelcolor='#ccc')
axes[2].grid(True, alpha=0.15, color='#666666')

plt.suptitle('Distribución Mixta (discreta + continua)', color='#ffffff', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/mixta.png', dpi=130, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print(f'P(X=0) empírico: {(X_mix==0).mean():.4f} | teórico: {p}')

---
## 4. Distribución de Bernoulli en base 3 generalizada

### Definición formal

Sea $p \in (0,1)$ con $p \neq 1/2$. Definimos:

$$X_p = \sum_{k=1}^{\infty} \frac{A_k}{3^k}, \quad A_k \in \{0, 2\}$$

donde $P(A_k = 2) = p$ y $P(A_k = 0) = 1-p$, con los $A_k$ i.i.d. La FDA de $X_p$ se construye igual que la escalera del diablo, pero los escalones tienen altura desigual: en el nivel $k$, la rama derecha recibe probabilidad $p$ y la izquierda $1-p$.

Para $p = 1/2$ recuperamos la distribución de Cantor clásica. Para $p \neq 1/2$, la distribución sigue siendo **singular continua** (apoyada en $\mathcal{C}$, sin densidad), pero la FDA ya no es simétrica.

**Momentos:**
$$\mathbb{E}[X_p] = \frac{2p}{3(1 - 1/3)} = \frac{2p}{2} = p, \quad \text{(se puede verificar por series geométricas)}$$

Más precisamente: $\mathbb{E}[X_p] = \frac{2p}{3} \cdot \frac{1}{1 - 1/3} \cdot \frac{1}{1} = p$.

### En criollo

> Es la misma construcción que Cantor, pero con una moneda cargada. Si $p = 0.8$, tendés a ir siempre hacia la derecha del conjunto de Cantor: la masa se acumula cerca del 1. La FDA sigue siendo una escalera, pero con escalones de alturas distintas a izquierda y derecha.

In [ ]:
def cantor_generalizado(N=20_000, K=18, p=0.5, seed=None):
    """Simula X_p = Σ Aₖ/3ᵏ con P(Aₖ=2)=p."""
    if seed is not None:
        np.random.seed(seed)
    A = 2 * np.random.binomial(1, p, size=(N, K))  # 0 o 2
    potencias = 3.0 ** (-np.arange(1, K + 1))
    return A @ potencias

ps = [0.2, 0.5, 0.8]
colores = ['#3498db', '#e8b96a', '#e74c3c']
N = 30_000

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
fig.patch.set_facecolor('#0f0f0f')
for row in axes:
    for ax in row:
        ax.set_facecolor('#1a1a1a')
        ax.tick_params(colors='#aaaaaa', labelsize=7)
        for spine in ax.spines.values():
            spine.set_edgecolor('#444444')

for i, (p_val, col) in enumerate(zip(ps, colores)):
    X = cantor_generalizado(N=N, K=16, p=p_val, seed=i)
    X_sorted = np.sort(X)
    y_ecdf = np.arange(1, N+1) / N

    # Histograma
    axes[0, i].hist(X, bins=300, density=True, color=col, alpha=0.8, edgecolor='none')
    axes[0, i].set_title(f'Histograma\np = {p_val}', color='#e8b96a', fontsize=9)
    axes[0, i].set_xlabel('x', color='#888888', fontsize=8)
    axes[0, i].set_xlim(0, 1)

    # FDA empírica
    axes[1, i].plot(X_sorted, y_ecdf, color=col, lw=0.9)
    axes[1, i].set_title(f'FDA empírica\np = {p_val}  |  E[X] ≈ {X.mean():.3f}',
                         color='#e8b96a', fontsize=9)
    axes[1, i].set_xlabel('x', color='#888888', fontsize=8)
    axes[1, i].set_ylabel('F̂(x)', color='#888888', fontsize=8)
    axes[1, i].grid(True, alpha=0.15, color='#666666')

plt.suptitle('Distribución de Cantor Generalizada — Bernoulli asimétrica en base 3',
             color='#ffffff', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/cantor_gen.png', dpi=130, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

---
## 5. Construcción via función de Minkowski (medida de la pregunta)

### Definición formal

La **función de Minkowski** $?(x)$ (pronunciada *"signo de pregunta de x"*) es una biyección de $[0,1]$ en $[0,1]$ que envía los irracionales cuadráticos a los números diádicos. Se define sobre las fracciones continuas:

Si $x = [0; a_1, a_2, a_3, \ldots]$ es la expansión en fracción continua de $x$, entonces:

$$?(x) = 2\sum_{k=1}^{\infty} \frac{(-1)^{k+1}}{2^{a_1 + a_2 + \cdots + a_k}}$$

Esta función es continua, estrictamente creciente, $?(0)=0$, $?(1)=1$, y satisface $?'(x)=0$ para casi todo $x$ (en medida de Lebesgue). Por lo tanto, define una **distribución de probabilidad singular continua** distinta de la de Cantor.

A diferencia de Cantor, el soporte de esta distribución es **todo** $[0,1]$ (no un conjunto de Cantor), pero igualmente carece de densidad.

### En criollo

> La función de Minkowski hace algo mágico: toma los números irracionales "más simples" (los cuadráticos, como $\sqrt{2}-1$ o el número áureo) y los lleva exactamente a los puntos diádicos $k/2^n$. La función crece en todos lados, pero su derivada es cero casi en todas partes: crece "de a saltos invisibles". Es otra forma de tener una distribución sin densidad, esta vez con soporte denso en $[0,1]$.

In [ ]:
def minkowski_question_mark(x, depth=40):
    """Calcula ?(x) via expansión en fracción continua."""
    results = []
    for xi in np.asarray(x).flatten():
        if xi <= 0: results.append(0.0); continue
        if xi >= 1: results.append(1.0); continue
        # Obtener coeficientes de fracción continua
        coeffs = []
        val = xi
        for _ in range(depth):
            a = int(val)
            coeffs.append(a)
            frac = val - a
            if frac < 1e-12:
                break
            val = 1.0 / frac
        # Calcular ?(x)
        s = 0.0
        exp = 0
        for k, a in enumerate(coeffs):
            exp += a
            s += (-1)**k / 2**exp
        results.append(2 * s)
    return np.array(results)

x_grid = np.linspace(0.001, 0.999, 2000)
y_mink = minkowski_question_mark(x_grid, depth=30)

# Algunos puntos notables
phi = (np.sqrt(5) - 1) / 2  # número áureo conjugado
sqrt2_frac = np.sqrt(2) - 1

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.patch.set_facecolor('#0f0f0f')
for ax in axes:
    ax.set_facecolor('#1a1a1a')
    ax.tick_params(colors='#aaaaaa', labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#444444')

# ?(x) vs x
axes[0].plot(x_grid, y_mink, color='#9b59b6', lw=1.5)
axes[0].plot([0, 1], [0, 1], color='#555555', lw=1, ls='--', label='y = x')
# Puntos notables
phi_q = float(minkowski_question_mark([phi]))
sqrt2_q = float(minkowski_question_mark([sqrt2_frac]))
axes[0].scatter([phi, sqrt2_frac], [phi_q, sqrt2_q],
                color=['#f1c40f', '#e74c3c'], zorder=5, s=40)
axes[0].annotate(f'φ={phi:.3f}\n?={phi_q:.3f}', (phi, phi_q),
                 xytext=(phi+0.1, phi_q-0.15), color='#f1c40f', fontsize=7)
axes[0].annotate(f'√2-1={sqrt2_frac:.3f}\n?={sqrt2_q:.3f}', (sqrt2_frac, sqrt2_q),
                 xytext=(sqrt2_frac-0.3, sqrt2_q+0.1), color='#e74c3c', fontsize=7)
axes[0].set_title('?(x): función de Minkowski', color='#e8b96a', fontsize=9)
axes[0].set_xlabel('x', color='#888888')
axes[0].set_ylabel('?(x)', color='#888888')
axes[0].legend(fontsize=7, facecolor='#2a2a2a', edgecolor='#555', labelcolor='#aaa')
axes[0].grid(True, alpha=0.15)

# Derivada numérica
dx = x_grid[1] - x_grid[0]
deriv = np.gradient(y_mink, dx)
axes[1].plot(x_grid, deriv, color='#e74c3c', lw=0.8, alpha=0.85)
axes[1].set_title("Derivada numérica de ?(x)\n(≈ 0 c.t.p.)", color='#e8b96a', fontsize=9)
axes[1].set_xlabel('x', color='#888888')
axes[1].set_ylabel("?'(x)", color='#888888')
axes[1].grid(True, alpha=0.15)
axes[1].set_ylim(-0.5, np.percentile(deriv, 99) * 1.5)

# Comparación con Cantor
axes[2].plot(x, fda_cantor, color='#e8b96a', lw=1.5, label='Cantor (soporte = C)', alpha=0.9)
axes[2].plot(x_grid, y_mink, color='#9b59b6', lw=1.5, label='Minkowski (soporte = [0,1])', alpha=0.9)
axes[2].plot([0, 1], [0, 1], color='#555555', lw=1, ls='--', label='Uniforme', alpha=0.6)
axes[2].set_title('Comparación de FDAs singulares', color='#e8b96a', fontsize=9)
axes[2].set_xlabel('x', color='#888888')
axes[2].set_ylabel('F(x)', color='#888888')
axes[2].legend(fontsize=7, facecolor='#2a2a2a', edgecolor='#555', labelcolor='#ccc')
axes[2].grid(True, alpha=0.15)

plt.suptitle('Función de Minkowski ?(x) — Distribución singular con soporte denso',
             color='#ffffff', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/minkowski.png', dpi=130, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

---
## Resumen comparativo

| Distribución | Soporte | FDA continua | Tiene densidad | Observación |
|---|---|---|---|---|
| **Cantor** | $\mathcal{C}$ (med. 0) | ✓ | ✗ | La clásica; escalera del diablo |
| **Cantor generalizada** ($p\neq 1/2$) | $\mathcal{C}$ (med. 0) | ✓ | ✗ | Escalones asimétricos |
| **Mixta** (discreta+continua) | $\{0\} \cup (0,\infty)$ | ✗ (salto en 0) | ✗ | Caso frecuente en aplicaciones |
| **Minkowski** $?(x)$ | $[0,1]$ (denso) | ✓ | ✗ | Soporte pleno, deriva de fracciones continuas |

### Mensaje central

La descomposición de Lebesgue garantiza que **toda** medida de probabilidad en $\mathbb{R}$ es combinación única de tres partes: absolutamente continua, singular continua y discreta pura. Los ejemplos anteriores ilustran que estas tres categorías coexisten y que la ausencia de densidad no implica discreción: existe un mundo continuo sin densidad, habitado por objetos matemáticamente ricos.